In [2]:
import os
import sys
import time
import dask
import zarr
import numpy as np
import xesmf as xe
import xarray as xr
import pandas as pd
from glob import glob

import shutil
from pathlib import Path

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
def hourly_datetimes(year: int) -> np.ndarray:
    start = np.datetime64(f"{year}-01-01T00:00:00", "ns")
    stop  = np.datetime64(f"{year+1}-01-01T00:00:00", "ns")  # exclusive
    hours = np.arange(start, stop, np.timedelta64(1, "h"))
    return hours  # dtype: datetime64[ns]

In [60]:
year = 1980
dt_list = hourly_datetimes(year)
flag_soil = True

In [61]:
base_dir = '/glade/campaign/ral/hap/ksha/DWC_data/CONUS_domain_GP/raw_404/'

varname_4d = ['WRF_P', 'WRF_Q', 'WRF_T', 'WRF_U', 'WRF_V', 'WRF_Q_tot']

if flag_soil:
    ds_static = xr.open_zarr(
        '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/static/C404_GP_static_LAKE.zarr'
    )

    land = ds_static["LANDMASK"]   # 1 = land, 0 = water
    lake = ds_static["LAKEMASK"]   # 1 = lake, 0 = land & ocean
    ocean_mask = (land == 0) & (lake == 0)  # True = ocean

    lat2d = ds_static["XLAT"]
    lon2d = ds_static["XLONG"]

    # ensure land mask coordinate names are south_north and west_east
    land_hw = land.rename({land.dims[-2]: "south_north", land.dims[-1]: "west_east"}) if land.dims != ("south_north", "west_east") else land
    lake_hw = lake.rename({lake.dims[-2]: "south_north", lake.dims[-1]: "west_east"}) if lake.dims != ("south_north", "west_east") else lake

    # ... ocean mask ...
    ocean_hw = ocean_mask.rename({ocean_mask.dims[-2]: "south_north", ocean_mask.dims[-1]: "west_east"}) if ocean_mask.dims != ("south_north", "west_east") else ocean_mask

    # ensure latd and lon2d xr.dataarrays have dim names of south_north and west_east
    lat_hw  = lat2d.rename({lat2d.dims[-2]: "south_north", lat2d.dims[-1]: "west_east"}) if lat2d.dims != ("south_north", "west_east") else lat2d
    lon_hw  = lon2d.rename({lon2d.dims[-2]: "south_north", lon2d.dims[-1]: "west_east"}) if lon2d.dims != ("south_north", "west_east") else lon2d

    # xESMF grid datasets (dims must match the data horizontal dims)
    src_grid = xr.Dataset(
        {
            "lat":  (("south_north", "west_east"), lat_hw.values),
            "lon":  (("south_north", "west_east"), lon_hw.values),
            "mask": (("south_north", "west_east"), land_hw.values),  # 1=keep, 0=ignore
        }
    )
    dst_grid = xr.Dataset(
        {
            "lat": (("south_north", "west_east"), lat_hw.values),
            "lon": (("south_north", "west_east"), lon_hw.values),
        }
    )

    regridder = xe.Regridder(
        src_grid,
        dst_grid,
        method="bilinear",
        extrap_method="nearest_s2d",
    ) # reuse_weights=True, filename="weights_conus404_fill.nc",

In [65]:
fn_year = sorted(glob(base_dir+f'*{year}*.zarr'))[:100] # <----- [:100]

if len(fn_year) > 0:
    file_collect = []
    
    for i_fn, fn in enumerate(fn_year):
        ds = xr.open_zarr(fn)
        ds['time'] = [dt_list[i_fn],]
        file_collect.append(ds)
        
    ds_year = xr.concat(file_collect, dim='time')

In [9]:
# merge all
ds_year = ds_year.drop_vars(['WRF_Q', 'WRF_Q_LC', 'WRF_PWAT_LC'], errors="ignore")

ds_year['WRF_precip_025'] = ds_year['WRF_precip']**0.25
ds_year['WRF_radar_composite_025'] = ds_year['WRF_radar_composite']**0.25
ds_year['WRF_PWAT_05'] = ds_year['WRF_PWAT']**0.5
ds_year['WRF_Q_tot_05'] = ds_year['WRF_Q_tot']**0.5

if flag_soil:
    # =================================================== #
    # SMOIS handling
    da_SMOIS = ds_year["WRF_SMOIS"]

    # mask out non-land; keep dims ("time","south_north","west_east")
    da_SMOIS_land = da_SMOIS.where(land_hw == 1)

    # FIX: regridder and data now share ("south_north","west_east") dims
    da_SMOIS_filled = regridder(da_SMOIS_land, skipna=True)

    # land vals corrected by original
    da_SMOIS_correct = xr.where(land_hw == 1, da_SMOIS, da_SMOIS_filled)

    # ocean vals forced to 0.0
    da_SMOIS_correct = xr.where(ocean_hw, 0.0, da_SMOIS_correct)

    da_SMOIS_correct = da_SMOIS_correct.transpose("time", "south_north", "west_east")
    ds_year["WRF_SMOIS"] = da_SMOIS_correct

    # =================================================== #
    # TSLB handling
    da_TSLB = ds_year["WRF_TSLB"]

    # treat lakes specially: mask out lakes (lake==1)
    da_TSLB_nolake = da_TSLB.where(lake_hw == 0)

    da_TSLB_filled = regridder(da_TSLB_nolake, skipna=True)

    # non-lake corrected by original; lakes replaced by filled
    da_TSLB_correct = xr.where(lake_hw == 0, da_TSLB, da_TSLB_filled)
    da_TSLB_correct = da_TSLB_correct.transpose("time", "south_north", "west_east")

    ds_year["WRF_TSLB"] = da_TSLB_correct

In [11]:
# ================ #
# check NaN
has_nan = ds_year.to_array().isnull().any().compute().item()
print("Has NaN:", has_nan)

Has NaN: False


In [ ]:
# =================================================== #
# rechunk
ds_year = ds_year.chunk(
    {
        'time': 16, 
        'bottom_top': 12, 
        'pressure_approx': 12, 
        'south_north': 336, 
        'west_east': 336
    }
)

varnames = list(ds_year.keys())
# zarr encodings
dict_encoding = {}

chunk_size_3d = dict(chunks=(16, 336, 336))
chunk_size_4d = dict(chunks=(16, 12, 336, 336))

compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)

for i_var, var in enumerate(varnames):
    if var in varname_4d:
        dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
    else:
        dict_encoding[var] = {'compressor': compress, **chunk_size_3d}

save_name = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404/C404_GP_{year}.zarr'
# ds_year.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
print(save_name)
print("--- %s seconds ---" % (time.time() - start_time))

## QC

In [11]:
year = 2023
ds1 = xr.open_zarr(f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404_new/C404_GP_{year}.zarr')
ds2 = xr.open_zarr(f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404/C404_GP_{year}.zarr')
GLW1 = ds1['WRF_P'].isel(time=299).values
GLW2 = ds2['WRF_P'].isel(time=299).values

In [9]:
ds = xr.open_zarr(f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404_new/C404_GP_{year}.zarr')

In [16]:
for year in range(2023, 2025):
    ds = xr.open_zarr(f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404_new/C404_GP_{year}.zarr')
    ds = ds.drop_vars(['XTIME'])
    has_nan = bool(ds.to_array().isnull().any().compute())
    if has_nan:
        print('{} has NaN'.format(year))
    else:
        print('{} good'.format(year))

2023 good
2024 good


In [6]:
ds1

<xarray.Dataset> Size: 467GB
Dimensions:                  (time: 8760, south_north: 336, west_east: 336,
                              bottom_top: 12, pressure_approx: 12)
Coordinates:
  * bottom_top               (bottom_top) float32 48B 0.0 1.0 2.0 ... 10.0 11.0
  * pressure_approx          (pressure_approx) float32 48B 1e+03 946.0 ... 100.0
  * south_north              (south_north) float32 1kB 0.0 1.0 ... 334.0 335.0
  * time                     (time) datetime64[ns] 70kB 2010-01-01 ... 2010-1...
  * west_east                (west_east) float32 1kB 0.0 1.0 2.0 ... 334.0 335.0
Data variables: (12/28)
    WRF_GLW                  (time, south_north, west_east) float32 4GB dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_MSLP                 (time, south_north, west_east) float32 4GB dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_OLR                  (time, south_north, west_east) float32 4GB dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_P                    (time, bottom_top, south_north, west_east) float32 47GB dask.array<chunksize=(16, 12, 336, 336), meta=np.ndarray>
    WRF_PWAT                 (time, south_north, west_east) float32 4GB dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_PWAT_05              (time, south_north, west_east) float32 4GB dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    ...                       ...
    WRF_Z                    (time, bottom_top, south_north, west_east) float32 47GB dask.array<chunksize=(16, 12, 336, 336), meta=np.ndarray>
    WRF_evapor               (time, south_north, west_east) float32 4GB dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_precip               (time, south_north, west_east) float32 4GB dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_precip_025           (time, south_north, west_east) float32 4GB dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_radar_composite      (time, south_north, west_east) float32 4GB dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_radar_composite_025  (time, south_north, west_east) float32 4GB dask.array<chunksize=(16, 336, 336), meta=np.ndarray>

In [ ]:
cp -r /glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404_new/C404_GP_2009.zarr/ .
cp -r /glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404_new/C404_GP_2010.zarr/ .

In [10]:
for year in range(2009, 2011):
    ds1 = xr.open_zarr(f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404_new/C404_GP_{year}.zarr')
    ds2 = xr.open_zarr(f'/glade/campaign/ral/hap/ksha/DWC_data/CONUS_domain_GP/C404_new/C404_GP_{year}.zarr')
    
    flag_match = ds1.identical(ds2)
    if flag_match:
        print(f'{year} ✓')
        continue
    else:
        raise ValueError(f'{year} mismatch between ds1 and ds2')

2009 ✓
2010 ✓


In [7]:
for year in range(1980, 2026):
    ds1 = xr.open_zarr(f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/dscale_1h/ERA5_GP_1h_{year}.zarr')
    ds2 = xr.open_zarr(f'/glade/campaign/ral/hap/ksha/DWC_data/CONUS_domain_GP/dscale_1h/ERA5_GP_1h_{year}.zarr')
    
    flag_match = ds1.identical(ds2)
    if flag_match:
        print(f'{year} ✓')
        continue
    else:
        raise ValueError(f'{year} mismatch between ds1 and ds2')

1980 ✓
1981 ✓
1982 ✓
1983 ✓
1984 ✓
1985 ✓
1986 ✓
1987 ✓
1988 ✓
1989 ✓
1990 ✓
1991 ✓
1992 ✓
1993 ✓
1994 ✓
1995 ✓
1996 ✓
1997 ✓
1998 ✓
1999 ✓
2000 ✓
2001 ✓
2002 ✓
2003 ✓
2004 ✓
2005 ✓
2006 ✓
2007 ✓
2008 ✓
2009 ✓
2010 ✓
2011 ✓
2012 ✓
2013 ✓
2014 ✓
2015 ✓
2016 ✓
2017 ✓
2018 ✓
2019 ✓
2020 ✓
2021 ✓
2022 ✓
2023 ✓
2024 ✓
2025 ✓


In [8]:
for year in range(1980, 2010):
    ds1 = xr.open_zarr(f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/dscale_CESM_HIST/CESM_GP_{year}.zarr')
    ds2 = xr.open_zarr(f'/glade/campaign/ral/hap/ksha/DWC_data/CONUS_domain_GP/dscale_CESM_HIST/CESM_GP_{year}.zarr')
    
    flag_match = ds1.identical(ds2)
    if flag_match:
        print(f'{year} ✓')
        continue
    else:
        raise ValueError(f'{year} mismatch between ds1 and ds2')

1980 ✓
1981 ✓
1982 ✓
1983 ✓
1984 ✓
1985 ✓
1986 ✓
1987 ✓
1988 ✓
1989 ✓
1990 ✓
1991 ✓
1992 ✓
1993 ✓
1994 ✓
1995 ✓
1996 ✓
1997 ✓
1998 ✓
1999 ✓
2000 ✓
2001 ✓
2002 ✓
2003 ✓
2004 ✓
2005 ✓
2006 ✓
2007 ✓
2008 ✓
2009 ✓


In [9]:
for year in range(2070, 2100):
    ds1 = xr.open_zarr(f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/dscale_CESM_SSP/CESM_GP_{year}.zarr')
    ds2 = xr.open_zarr(f'/glade/campaign/ral/hap/ksha/DWC_data/CONUS_domain_GP/dscale_CESM_SSP/CESM_GP_{year}.zarr')
    
    flag_match = ds1.identical(ds2)
    if flag_match:
        print(f'{year} ✓')
        continue
    else:
        raise ValueError(f'{year} mismatch between ds1 and ds2')

2070 ✓
2071 ✓
2072 ✓
2073 ✓
2074 ✓
2075 ✓
2076 ✓
2077 ✓
2078 ✓
2079 ✓
2080 ✓
2081 ✓
2082 ✓
2083 ✓
2084 ✓
2085 ✓
2086 ✓
2087 ✓
2088 ✓
2089 ✓
2090 ✓
2091 ✓
2092 ✓
2093 ✓
2094 ✓
2095 ✓
2096 ✓
2097 ✓
2098 ✓
2099 ✓
